In [1]:
# Importamos las librerías básicas

import numpy as np
import pandas as pd

# modificamos la configuración para ver todas las columnas al mostrar dataframes
pd.set_option("display.max_columns", None)


In [2]:
# Carga de datasets

# Cargar dataset CSV (campañas de marketing)
bank_df = pd.read_csv( "../Data/DataRaw/bank-additional.csv", sep=",", index_col=0)  # usamos coma como separador

# Cargar dataset Excel (detalles de clientes con varias hojas)
customer_xlsx = pd.ExcelFile("../Data/DataRaw/customer-details.xlsx")
customer_2012 = pd.read_excel(customer_xlsx, sheet_name="2012", index_col=0)
customer_2013 = pd.read_excel(customer_xlsx, sheet_name="2013", index_col=0)
customer_2014 = pd.read_excel(customer_xlsx, sheet_name="2014", index_col=0)

# Utilizamos index_col=0 para indicar que la primera columna se use como índice

In [3]:
bank_df.sample(5)

,age,job,marital,education,default,housing,loan,contact,duration,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y,date,latitude,longitude,id_
11887,35.0,admin.,MARRIED,university.degree,0.0,0.0,0.0,telephone,13,4,999,0,NONEXISTENT,1.4,"94,465","-41,8",NaN,"5228,1",no,27-diciembre-2016,39.821,-76.547,7aee25a7-cb30-45fa-88be-1372f06b2ccd
15203,31.0,services,SINGLE,high.school,0.0,0.0,0.0,cellular,186,2,999,0,NONEXISTENT,1.4,"93,918","-42,7","4,958","5228,1",no,28-marzo-2018,29.833,-67.398,a44a6c2e-a0fc-43a1-9d7f-995d1b5ed984
9634,39.0,management,MARRIED,university.degree,0.0,1.0,0.0,telephone,23,1,999,0,NONEXISTENT,1.4,"94,465","-41,8",NaN,"5228,1",no,17-febrero-2019,31.433,-95.138,4ade6948-e26e-464e-a8e0-df05284aeb3f
38539,NaN,retired,MARRIED,high.school,0.0,0.0,0.0,cellular,207,1,6,1,SUCCESS,-3.4,"92,431","-26,9","0,724","5017,5",yes,8-abril-2015,26.622,-88.786,c09339b8-85b9-4e4e-a22e-62c12008c20a
36788,33.0,unemployed,MARRIED,university.degree,0.0,1.0,0.0,cellular,579,3,999,0,NONEXISTENT,-2.9,"92,963","-40,8","1,26","5076,2",yes,6-septiembre-2016,38.697,-79.713,f3ba5053-f021-4f4b-ab80-8b5dd03ff881


Antes de realizar las transformaciones vamos a hacer una copia de nuestro set de datos para trabajar con ella. La primera transformación va a ser pasar la columna age de float a int ya que los valores son edades de los clientes que deben ser enteros.

In [4]:
df_bank_copy = bank_df.copy()

In [5]:
# Aplicamos lambda para convertir a int, cuidando los NaN
df_bank_copy['age'] = df_bank_copy['age'].apply(lambda x: int(x) if pd.notnull(x) else pd.NA)

# Convertimos a tipo entero de pandas (Int64) para mantener los NaN
df_bank_copy['age'] = df_bank_copy['age'].astype('Int64')

# Comprobamos
print("-> age dtype:", df_bank_copy['age'].dtype)


-> age dtype: Int64


Vamos a normalizar las tres columnas booleanas (default, housing, loan) usando map.
Haremos el reemplazo 0 → "no" y 1 → "yes", para tener la misma nomenclatura que la columna y.

In [6]:
# Normalizamos columnas booleanas con map (0 -> 'no', 1 -> 'yes')

bool_cols = ['default', 'housing', 'loan']

for col in bool_cols:
    df_bank_copy[col] = df_bank_copy[col].map({0: 'no', 1: 'yes'})
    print(f"-> {col}: valores únicos tras normalización:", df_bank_copy[col].unique())
    # mostramos los valores únicos de cada columna para comprobar que solo 
    # quedan "no" y "yes" (y NaN si existieran).


-> default: valores únicos tras normalización: ['no' nan 'yes']
-> housing: valores únicos tras normalización: ['no' 'yes' nan]
-> loan: valores únicos tras normalización: ['no' 'yes' nan]


In [7]:
df_bank_copy[bool_cols].sample(5)

,default,housing,loan
22156,no,no,no
7706,no,yes,no
28562,no,no,no
16691,no,no,no
2976,no,yes,no


Vamos a convertir las columnas:
- 'cons.price.idx'
- 'cons.conf.idx'
- 'euribor3m'
- 'nr.employed'

De tipo object -> float

aplicamos .str.replace(',', '.') para pasar las comas decimales a puntos.

In [8]:
# Convertimos columnas 'object' con comas decimales a float

conv_float = ['cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed']

for col in conv_float:
    if df_bank_copy[col].dtype == 'object': #nos aseguramos que es tipo obj
        # Reemplazar comas por puntos y convertir a float
        df_bank_copy[col] = df_bank_copy[col].str.replace(',', '.', regex=False).astype(float)
        print(f"-> {col} convertido a float")
    else:
        print(f"-> {col} ya es {df_bank_copy[col].dtype}, no necesita conversión")
    
#Comprobamos que ahora son tipo float
display(df_bank_copy[conv_float].sample(5))
df_bank_copy[conv_float].dtypes


-> cons.price.idx convertido a float
-> cons.conf.idx convertido a float
-> euribor3m convertido a float
-> nr.employed convertido a float


,cons.price.idx,cons.conf.idx,euribor3m,nr.employed
3617,93.994,-36.4,4.859,5191.0
27952,92.843,-50.0,1.520,5099.1
39125,92.713,-33.0,0.700,5023.5
5521,93.994,-36.4,4.857,5191.0
23298,93.444,-36.1,NaN,5228.1


cons.price.idx    float64
cons.conf.idx     float64
euribor3m         float64
nr.employed       float64
dtype: object

Reconsideramos que la columna nr.employed debería ser int ya que no tiene mucho sentido tener un número decimal de empleados. 

In [9]:
# Conversión de nr.employed a entero
    
# Truncamos a int (manteniendo NaN donde corresponda)
df_bank_copy['nr.employed'] = df_bank_copy['nr.employed'].apply(lambda x: int(x) if pd.notnull(x) else pd.NA)
    
# Convertimos a tipo entero de pandas (Int64) para mantener los NaN
df_bank_copy['nr.employed'] = df_bank_copy['nr.employed'].astype('Int64')
    
print("-> 'nr.employed' convertido a entero Int64")
df_bank_copy['nr.employed'].sample(5)


-> 'nr.employed' convertido a entero Int64


27282    5195
13175    5228
35487    5099
22804    5228
8965     5228
Name: nr.employed, dtype: Int64

Convertimos columna date de object a datetime

In [10]:
df_bank_copy['date'].sample(5)

14535       14-marzo-2019
32033    5-noviembre-2018
31393       15-abril-2016
35860       27-julio-2018
38660       5-agosto-2019
Name: date, dtype: object

Mapeamos manualmente los meses en español y los convertimos a números

In [ ]:
meses = {
    "enero":"01","febrero":"02","marzo":"03","abril":"04",
    "mayo":"05","junio":"06","julio":"07","agosto":"08",
    "septiembre":"09","octubre":"10","noviembre":"11","diciembre":"12"
}

#Mapeamos de forma manual, cambiamos meses en letra por su correspondiente número de mes
for mes, num in meses.items():
    df_bank_copy['date'] = df_bank_copy['date'].replace(mes, num, regex=False)

#Convertimos a datetime
df_bank_copy['date'] = pd.to_datetime(df_bank_copy['date'], dayfirst=True)

#Hacemos un sample para comprobar
df_bank_copy['date'].sample(5)

3697    2018-06-08
20782   2015-12-24
15341   2017-07-25
23978   2016-07-06
35981   2016-06-04
Name: date, dtype: datetime64[ns]

Eliminamos columnas laitude y longitude por no aportar información relevante para nuestro EDA

In [25]:
# Comprobamos que existen las columnas antes de eliminarlas
cols_to_drop = ["latitude", "longitude"]
cols_eliminar = [col for col in cols_to_drop if col in df_bank_copy.columns]

# Eliminamos las columnas presentes
df_bank_copy = df_bank_copy.drop(columns=cols_eliminar)

# Comprobamos el resultado
print("Columnas actuales de df_bank_copy:", df_bank_copy.columns)
df_bank_copy.shape


Columnas actuales de df_bank_copy: Index(['age', 'job', 'marital', 'education', 'default', 'housing', 'loan',
       'contact', 'duration', 'campaign', 'pdays', 'previous', 'poutcome',
       'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m',
       'nr.employed', 'y', 'date', 'id_'],
      dtype='object')


(43000, 21)

Nos falta crear las columnas contact_month y contact_year a partir de date